## Primal Formulation

For a linearly separable dataset $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $y_i \in \{-1, +1\}$,
the hard-margin SVM solves:

$$
\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2
\quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 \quad \forall i
$$

The soft-margin extension introduces slack variables $\xi_i \geq 0$:

$$
\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^n \xi_i
\quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i
$$

where $C > 0$ controls the bias-variance tradeoff.


## Dual Formulation

Forming the Lagrangian and applying KKT conditions yields the dual:

$$
\max_{\boldsymbol{\alpha}} \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^\top \mathbf{x}_j
\quad \text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_i \alpha_i y_i = 0
$$

The decision function becomes:

$$
f(\mathbf{x}) = \text{sign}\!\left(\sum_{i \in \mathcal{S}} \alpha_i y_i \kappa(\mathbf{x}_i, \mathbf{x}) + b\right)
$$

where $\mathcal{S}$ is the support vector set and $\kappa$ is the kernel function.

For the RBF kernel:

$$
\kappa(\mathbf{x}, \mathbf{x}') = \exp\!\left(-\gamma\|\mathbf{x} - \mathbf{x}'\|^2\right)
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import sys
sys.path.insert(0, '..')
from assets.matplotlib_theme import apply_void_theme, vc
apply_void_theme('dark')

# Synthetic 2D dataset — visualize decision boundary
np.random.seed(0)
n = 150
X_pos = np.random.randn(n, 2) + [2, 2]
X_neg = np.random.randn(n, 2) + [-2, -2]
X = np.vstack([X_pos, X_neg])
y = np.array([1]*n + [-1]*n)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

svm = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
svm.fit(X_s, y)

# Decision boundary grid
xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 300), np.linspace(-3.5, 3.5, 300))
Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, levels=[-999, 0, 999],
            colors=[vc.red, vc.cyan], alpha=0.08)
ax.contour(xx, yy, Z, levels=[-1, 0, 1],
           colors=[vc.red, vc.text, vc.cyan],
           linewidths=[0.8, 1.4, 0.8],
           linestyles=['--', '-', '--'])

ax.scatter(X_s[y==1, 0], X_s[y==1, 1], c=vc.cyan, s=18, alpha=0.6, label='+1')
ax.scatter(X_s[y==-1, 0], X_s[y==-1, 1], c=vc.red, s=18, alpha=0.6, label='-1')
ax.scatter(X_s[svm.support_, 0], X_s[svm.support_, 1],
           s=80, facecolors='none', edgecolors=vc.yellow, lw=1.2, label='support vectors')

ax.set_title('RBF SVM Decision Boundary — C=1.0')
ax.set_xlabel('$x_1$ (scaled)')
ax.set_ylabel('$x_2$ (scaled)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Support vectors: {svm.n_support_} (class -1 / class +1)")
print(f"Train accuracy: {svm.score(X_s, y):.4f}")